In [ ]:
# Download processed arabidopsis annotation generated by G. Benegas, et al.
# from https://huggingface.co/datasets/gonzalobenegas/processed-data-arabidopsis
!mkdir -p data
!wget -c -O data/windows.parquet "https://hf-mirror.com/datasets/gonzalobenegas/processed-data-arabidopsis/resolve/main/embedding/windows.parquet?download=true"

In [1]:
import random
import pandas as pd
from pyfastx import Fasta
import torch

In [ ]:
# Extract sequences and labels
windows = pd.read_parquet("data/windows.parquet")
fasta = Fasta("data/TAIR10.fa.gz")
genome = {chrom.name: chrom.seq for chrom in fasta}
sequences = [[], []]
labels = []
label_names = {}
for row in windows.itertuples(index=False):
    chrom = "Chr" + row[0]
    start = row[1]
    end = row[2]
    seq_fwd = genome[chrom][start:end].upper()
    seq_rev = seq_fwd.translate(str.maketrans("ACGTN", "TGCAN"))[::-1]
    label = row[-1]
    sequences[0].append(seq_fwd)
    sequences[1].append(seq_rev)
    if label not in label_names:
        label_names[label] = len(label_names)
    labels.append(label_names[label])

# (Optional) randomly sample sequences
sample_size = 500000
random.seed(42)
combined = list(zip(sequences[0], sequences[1], labels))
random.shuffle(combined)
sequences[0][:], sequences[1][:], labels[:] = zip(*combined)
sequences = [sequences[0][:sample_size], sequences[1][:sample_size]]
labels = labels[:sample_size]

In [2]:
from dnallm import load_config
from dnallm import load_model_and_tokenizer, DNAInference

In [3]:
# Load configurations
configs = load_config("./inference_config.yaml")
configs["inference"].max_length = 512
configs["inference"].batch_size = 64

In [ ]:
model_name = "lgq12697/gpn-brassicales"
model, tokenizer = load_model_and_tokenizer(model_name, task_config=configs["task"], source="modelscope")

20:47:14 - dnallm.models.model - INFO - Model files are stored in /home/liuguanqing/.cache/modelscope/hub/models/lgq12697/gpn-brassicales


In [ ]:
# Create inference engine
inference_engine = DNAInference(
    model=model,
    tokenizer=tokenizer,
    config=configs
)

20:47:14 - dnallm.models.model - INFO - Using device: cuda


In [ ]:
# Get embeddings for forward and reverse sequences
embeddings = inference_engine.get_embeddings(sequences[0], do_reduce=True, reduce_strategy=100, force=True)
embeddings_rc = inference_engine.get_embeddings(sequences[1], do_reduce=True, reduce_strategy=100, force=True)

# Average embeddings from forward and reverse sequences
inference_engine.embeddings["hidden_states"] = tuple(
    [(embeddings[i] + embeddings_rc[i]) / 2
     for i in range(len(embeddings))]
)

# Set labels for plotting
inference_engine.embeddings["labels"] = torch.tensor(labels)
inference_engine.task_config.label_names = list(label_names.keys())

Encoding inputs:   0%|          | 0/500000 [00:00<?, ? examples/s]

Inferring: 100%|██████████| 7813/7813 [17:52<00:00,  7.29it/s]


Encoding inputs:   0%|          | 0/500000 [00:00<?, ? examples/s]

Inferring: 100%|██████████| 7813/7813 [16:37<00:00,  7.84it/s]


In [ ]:
# (Optional) Save embeddings for future use
torch.save(inference_engine.embeddings, "figure3g_embeddings.pt")
# Load embeddings from local file
# inference_engine.embeddings = torch.load("figure3g_embeddings.pt")

In [ ]:
# Plot embeddings using custom dimensionality reduction method
# quality = {
#     "n_neighbors": 15,
#     "min_dist": 0.1,
#     "n_components": 2,
#     "metric": "euclidean"
# }
quality = "high"
plot = inference_engine.plot_hidden_states(point_size=2, reduced=True, reducer="umap", quality=quality)
plot.save("Figure3g.svg")